Copias impresas y electrónicas de *Modelado y simulación en Python* están disponibles en [No Starch Press](https://nostarch.com/modeling-and-simulation-python) y [Bookshop.org](https://bookshop.org/p/books/modeling-and-simulation-in-python-allen-b-downey/17836697?ean=9781718502161) y [Amazon](https://amzn.to/3y9UxNb).

# El Empire State Building contraataca

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [18]:
# install Pint if necessary

try:
    from pint import UnitRegistry
except ImportError:
    !pip install pint
    
# import units
from pint import UnitRegistry
units = UnitRegistry()

In [19]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://raw.githubusercontent.com/AllenDowney/' +
         'ModSimPy/master/modsim.py')

In [20]:
# import functions from modsim

from modsim import *

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Haga clic aquí para acceder a los cuadernos: <https://allendowney.github.io/ModSimPy/>.

Hasta ahora, las ecuaciones diferenciales con las que hemos trabajado han sido de *primer orden*, lo que significa que solo involucran primeras derivadas. en esto
En este capítulo, centramos nuestra atención en las ecuaciones diferenciales de *segundo orden*, que pueden incluir derivadas tanto de primera como de segunda.

Revisaremos el ejemplo de la moneda que cae del Capítulo 1 y usaremos `run_solve_ivp` para encontrar la posición y la velocidad de la moneda mientras cae, con y sin resistencia del aire.

## Segunda ley del movimiento de Newton

Las ecuaciones diferenciales (ED) de primer orden se pueden escribir 

$$\frac{dy}{dx} = G(x, y)$$ 

donde $G$ es alguna función de $x$ y $y$ (ver <http://modsimpy.com/ode>). Los DE de segundo orden se pueden escribir 

$$\frac{d^2y}{dx^2} = H(x, y, \frac{dy}{dt})$$

donde $H$ es una función de $x$, $y$ y $dy/dx$.

En este capítulo, trabajaremos con uno de los más famosos y útiles.
DE de segundo orden, segunda ley del movimiento de Newton: 

$$F = m a$$ 

donde $F$ es una fuerza o el total de un conjunto de fuerzas, $m$ es la masa de un objeto en movimiento y $a$ es su aceleración.

La ley de Newton podría no parecer una ecuación diferencial, hasta que sepamos
darse cuenta de que la aceleración, $a$, es la segunda derivada de la posición,
$y$, respecto al tiempo, $t$. con la sustitucion

$$a = \frac{d^2y}{dt^2}$$ 

La ley de Newton se puede escribir.

$$\frac{d^2y}{dt^2} = F / m$$ 

Y ese es definitivamente un DE de segundo orden.
En general, $F$ puede ser función del tiempo, la posición y la velocidad.

Por supuesto, esta "ley" es realmente un modelo en el sentido de que es una
Simplificación del mundo real. Aunque a menudo es aproximadamente
cierto:

- Sólo aplica si $m$ es constante. Si la masa depende del tiempo,
    posición o velocidad, tenemos que usar una forma más general de
    Ley de Newton (ver <http://modsimpy.com/varmass>).

- No es un buen modelo para cosas muy pequeñas, que son mejores
    descrito por otro modelo, la mecánica cuántica.

- Y no es un buen modelo para cosas que se mueven muy rápido, que son
    descrito mejor por otro modelo más, la mecánica relativista.

Sin embargo, para cosas de tamaño mediano con masa constante, que se mueven a
velocidades medias, el modelo de Newton es extremadamente útil. si podemos
cuantificar las fuerzas que actúan sobre tal objeto, podemos predecir cómo
se moverá.

## Dejar caer centavos

Como primer ejemplo, volvamos al centavo que cae del Empire State Building, que consideramos en el Capítulo 1. Implementaremos dos modelos de este sistema: primero sin resistencia del aire, luego con.

Dado que el Empire State Building tiene 381 m de altura, y suponiendo que
el centavo se deja caer desde un punto muerto, las condiciones iniciales son:

In [21]:
init = State(y=381, v=0)

donde `y` es la altura sobre la acera y `v` es la velocidad. 

Pondré las condiciones iniciales en un objeto `System`, junto con la magnitud de la aceleración debida a la gravedad, `g`, y la duración de las simulaciones, `t_end`.

In [22]:
system = System(init=init, 
                g=9.8, 
                t_end=10)

.
Ahora necesitamos una función de pendiente, y aquí es donde las cosas se ponen complicadas. Como hemos visto, `run_solve_ivp` puede resolver sistemas de ED de primer orden, pero la ley de Newton es una ED de segundo orden. Sin embargo, si reconocemos que

1. La velocidad, $v$, es la derivada de la posición, $dy/dt$, y

2. La aceleración, $a$, es la derivada de la velocidad, $dv/dt$,

Podemos reescribir la ley de Newton como un sistema de EDO de primer orden:

$$\frac{dy}{dt} = v$$ 

$$\frac{dv}{dt} = a$$ 

Y podemos traducirlos
ecuaciones en una función de pendiente:

In [23]:
def slope_func(t, state, system):
    y, v = state

    dydt = v
    dvdt = -system.g
    
    return dydt, dvdt

Como es habitual, los parámetros son una marca de tiempo, un objeto `State` y un objeto `System`.

La primera línea descomprime las variables de estado, `y` y `v`.

Las dos líneas siguientes calculan las derivadas de las variables de estado, `dydt` y `dvdt`.
La derivada de la posición es la velocidad y la derivada de la velocidad es la aceleración.
En este caso, $a = -g$, lo que indica que la aceleración debida a la gravedad es en dirección decreciente $y$. 

`slope_func` devuelve una secuencia que contiene las dos derivadas.

Antes de llamar a `run_solve_ivp`, es una buena idea probar la pendiente
funcionar con las condiciones iniciales:

In [24]:
dydt, dvdt = slope_func(0, system.init, system)
dydt, dvdt

El resultado es 0 m/s para velocidad y -9,8 m/s$^2$ para aceleración.

Ahora llamamos a `run_solve_ivp` así:

In [25]:
results, details = run_solve_ivp(system, slope_func)
details.message

`results` es un `TimeFrame` con dos columnas: `y` contiene la altura del centavo; `v` contiene su velocidad.
Aquí están las primeras filas.

In [26]:
results.head()

Podemos trazar los resultados de esta manera:

In [27]:
results.y.plot()

decorate(xlabel='Time (s)',
         ylabel='Position (m)')

Como la aceleración es constante, la velocidad aumenta linealmente y la posición disminuye cuadráticamente; como resultado, la curva de altura es una parábola.

El último valor de `results.y` es negativo, lo que significa que ejecutamos la simulación por demasiado tiempo. 

In [28]:
results.iloc[-1].y

Una forma de resolver este problema es utilizar los resultados para
estima el momento en que la moneda llega a la acera.

La biblioteca ModSim proporciona `crossings`, que toma un `TimeSeries` y un valor, y devuelve una secuencia de veces cuando la serie pasa por el valor. Podemos encontrar el momento en el que la altura del centavo es `0` así:

In [29]:
t_crossings = crossings(results.y, 0)
t_crossings

El resultado es una matriz con un único valor, 8,818 s. Ahora podríamos correr
la simulación nuevamente con `t_end = 8.818`, pero hay una manera mejor.

## Eventos

Como opción, `run_solve_ivp` puede tomar una *función de evento*, que
detecta un "evento", como el centavo que golpea la acera, y finaliza el
simulación.

Las funciones de evento toman los mismos parámetros que las funciones de pendiente, `t`, `state` y `system`. Deben devolver un valor que pase por `0` cuando ocurre el evento. Aquí hay una función de evento que detecta el centavo que golpea la acera:

In [30]:
def event_func(t, state, system):
    y, v = state
    return y

El valor de retorno es la altura del centavo, `y`, que pasa por
`0` cuando el centavo llega a la acera.

Pasamos la función de evento a `run_solve_ivp` así:

In [31]:
results, details = run_solve_ivp(system, slope_func,
                                 events=event_func)
details.message

Entonces podemos obtener el tiempo de vuelo así:

In [32]:
t_end = results.index[-1]
t_end

Y la velocidad final así: 

In [33]:
y, v = results.iloc[-1]
y, v

Si no hubiera resistencia del aire, la moneda golpearía la acera (o la cabeza de alguien) a aproximadamente 86 m/s. Por eso es bueno que haya resistencia al aire.

## Resumen

En este capítulo, escribimos la segunda ley de Newton, que es una ED de segundo orden, como un sistema de ED de primer orden.
Luego utilizamos `run_solve_ivp` para simular la caída de un centavo desde el Empire State Building en ausencia de resistencia del aire.
Y usamos una función de evento para detener la simulación cuando el centavo llega a la acera.

En el próximo capítulo agregaremos resistencia del aire al modelo.
Pero primero quizás quieras trabajar en este ejercicio.

## Ejercicios

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Puede acceder a los cuadernos en <https://allendowney.github.io/ModSimPy/>.

### Ejercicio 1

Aquí hay una pregunta del sitio web *Pregúntele a un astrónomo* (ver http://curious.astro.cornell.edu/about-us/39-our-solar-system/the-earth/other-catastrophes/57-how-long-would-it-take-the-earth-to-fall-into-the-sun-intermediate):

> "Si la Tierra repentinamente dejara de orbitar alrededor del Sol, sé que eventualmente sería atraída por la gravedad del Sol y lo golpearía. ¿Cuánto tiempo le tomaría a la Tierra golpear al Sol? Me imagino que iría lentamente al principio y luego ganaría velocidad".

Utilice `run_solve_ivp` para responder esta pregunta.

Aquí hay algunas sugerencias sobre cómo proceder:

1. Busque la Ley de Gravitación Universal y las constantes que necesite.  Te sugiero que trabajes completamente en unidades SI: metros, kilogramos y Newtons.

2. Cuando la distancia entre la Tierra y el Sol se reduce, este sistema se comporta mal, por lo que debes usar una función de evento para detenerlo cuando la superficie de la Tierra llegue a la superficie del Sol.

3. Exprese su respuesta en días y represente los resultados como millones de kilómetros versus días.

Si lee la respuesta de Dave Rothstein, verá otras formas de resolver el problema y una buena discusión sobre las decisiones de modelado detrás de ellas.

Quizás también te interese saber que no es tan fácil llegar al Sol; ver https://www.theatlantic.com/science/archive/2018/08/parker-solar-probe-launch-nasa/567197/.

In [34]:
# Solution goes here

In [35]:
# Solution goes here

In [36]:
# Solution goes here

In [37]:
# Solution goes here

In [38]:
# Solution goes here

In [39]:
# Solution goes here

In [40]:
# Solution goes here

In [41]:
# Solution goes here

In [42]:
# Solution goes here

In [43]:
# Solution goes here

In [44]:
# Solution goes here

In [45]:
# Solution goes here

In [46]:
# Solution goes here

In [47]:
# Solution goes here

In [48]:
# Solution goes here